In [3]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error

import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error

# ==========================================
# [1] LOAD DATA (PERBAIKAN ENCODING)
# ==========================================

filename = 'musicgenre-small.csv'

try:
    df = pd.read_csv(filename, delimiter=';', encoding='latin-1')
    print(f"Sukses! File '{filename}' ditemukan.")

    # Membersihkan data: Mengubah format angka dari string (koma) ke float (titik)
    cols_to_convert = ['loudness', 'tempo', 'duration'] + \
                      [f'avg_timbre{i}' for i in range(1, 13)] + \
                      [f'var_timbre{i}' for i in range(1, 13)]

    for col in cols_to_convert:
        if df[col].dtype == 'object':
            try:
                df[col] = df[col].astype(str).str.replace(',', '.').astype(float)
            except ValueError:
                pass

    print(f"Data berhasil dimuat! Dimensi: {df.shape}")
    print("-" * 30)

except FileNotFoundError:
    print(f"ERROR: File '{filename}' tidak ditemukan!")
    print("Pastikan Anda sudah upload file ke menu Folder di sebelah kiri Colab.")
except UnicodeDecodeError:
    print("Mencoba encoding alternatif...")
    df = pd.read_csv(filename, delimiter=';', encoding='cp1252')
    print("Sukses dengan encoding cp1252!")

# ==========================================
# [2] DESKRIPSI STATISTIK (Untuk Poin 1)
# ==========================================
print("Statistik Deskriptif Ringkas:")
print(df[['loudness', 'tempo', 'duration']].describe())
print("-" * 30)

# ==========================================
# [3] MEMBANGUN MODEL REGRESI (Untuk Poin 3)
# ==========================================

y = df['loudness']

# --- MODEL A: Fitur Dasar ---
X_A = df[['tempo', 'duration', 'time_signature', 'key', 'mode']]
X_A = sm.add_constant(X_A)
model_a = sm.OLS(y, X_A).fit()
rmse_a = np.sqrt(mean_squared_error(y, model_a.predict(X_A)))

# --- MODEL B: Fitur Timbre (2-12) ---
timbre_cols = [f'avg_timbre{i}' for i in range(2, 13)]
X_B = df[timbre_cols]
X_B = sm.add_constant(X_B)
model_b = sm.OLS(y, X_B).fit()
rmse_b = np.sqrt(mean_squared_error(y, model_b.predict(X_B)))

# --- MODEL C: Gabungan ---
X_C = pd.concat([df[['tempo', 'duration', 'time_signature', 'key', 'mode']], df[timbre_cols]], axis=1)
X_C = sm.add_constant(X_C)
model_c = sm.OLS(y, X_C).fit()
rmse_c = np.sqrt(mean_squared_error(y, model_c.predict(X_C)))

# ==========================================
# [4] HASIL & PERBANDINGAN
# ==========================================

print("\n=== PERBANDINGAN MODEL ===")
print(f"Model A (Basic)    -> R-squared: {model_a.rsquared:.4f} | RMSE: {rmse_a:.4f}")
print(f"Model B (Timbre)   -> R-squared: {model_b.rsquared:.4f} | RMSE: {rmse_b:.4f}")
print(f"Model C (Combined) -> R-squared: {model_c.rsquared:.4f} | RMSE: {rmse_c:.4f}")

print("\n=== DETAIL MODEL TERBAIK (MODEL C) ===")
print(model_c.summary())

Sukses! File 'musicgenre-small.csv' ditemukan.
Data berhasil dimuat! Dimensi: (18492, 33)
------------------------------
Statistik Deskriptif Ringkas:
           loudness         tempo      duration
count  18492.000000  18492.000000  18492.000000
mean     -11.839583    121.561750    273.836043
std        6.291408     36.328381    149.846992
min      -48.057000      0.000000      2.272200
25%      -14.871500     95.800000    193.384040
50%      -10.547500    119.965500    242.886080
75%       -7.176750    142.112750    320.639550
max        2.865000    256.663000   2873.808530
------------------------------

=== PERBANDINGAN MODEL ===
Model A (Basic)    -> R-squared: 0.0544 | RMSE: 6.1178
Model B (Timbre)   -> R-squared: 0.5051 | RMSE: 4.4259
Model C (Combined) -> R-squared: 0.5133 | RMSE: 4.3892

=== DETAIL MODEL TERBAIK (MODEL C) ===
                            OLS Regression Results                            
Dep. Variable:               loudness   R-squared:                       0